In [3]:
import tensorflow as tf
from tensorflow.keras.models import load_model
import pickle
import pandas as pd
import numpy as np


In [4]:
#load the trained model
model = load_model('model.h5')

#load the encoders
with open('label_encoder_gender.pkl','rb') as file:
    label_encoder_gender = pickle.load(file)

#load the one hot encoder for geography
with open('one_hot_encoder_geo.pkl','rb') as file:
    one_hot_encoder_geo = pickle.load(file)

#load scaler for numerical features
with open('scaler.pkl','rb') as file:
    scaler = pickle.load(file)

In [21]:
# Example input data
input_data = {
    'CreditScore': 600,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 40,
    'Tenure': 3,
    'Balance': 60000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000
}

In [22]:
# apply label encoding to this input data
input_data['Gender'] = label_encoder_gender.transform([input_data['Gender']])[0]
print(input_data)



{'CreditScore': 600, 'Geography': 'France', 'Gender': 1, 'Age': 40, 'Tenure': 3, 'Balance': 60000, 'NumOfProducts': 2, 'HasCrCard': 1, 'IsActiveMember': 1, 'EstimatedSalary': 50000}


In [23]:
## apply one hot encoding to the geography feature
geo_encoded = one_hot_encoder_geo.transform([[input_data['Geography']]]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded, columns=one_hot_encoder_geo.get_feature_names_out(['Geography']))


d:\MindsprintCourse\dl\ann\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


In [24]:
print(geo_encoded_df)

   Geography_France  Geography_Germany  Geography_Spain
0               1.0                0.0              0.0


In [25]:
print(input_data)

{'CreditScore': 600, 'Geography': 'France', 'Gender': 1, 'Age': 40, 'Tenure': 3, 'Balance': 60000, 'NumOfProducts': 2, 'HasCrCard': 1, 'IsActiveMember': 1, 'EstimatedSalary': 50000}


In [26]:
# remove the original geography column and add the one hot encoded columns for geography
input_data.pop('Geography')
input_data.update(geo_encoded_df.iloc[0].to_dict())

In [27]:
print(input_data)

{'CreditScore': 600, 'Gender': 1, 'Age': 40, 'Tenure': 3, 'Balance': 60000, 'NumOfProducts': 2, 'HasCrCard': 1, 'IsActiveMember': 1, 'EstimatedSalary': 50000, 'Geography_France': 1.0, 'Geography_Germany': 0.0, 'Geography_Spain': 0.0}


In [28]:
# check the type of input data
print(type(input_data))

<class 'dict'>


In [29]:
input_df = pd.DataFrame([input_data])

In [30]:
print(input_df)

   CreditScore  Gender  Age  Tenure  Balance  NumOfProducts  HasCrCard  \
0          600       1   40       3    60000              2          1   

   IsActiveMember  EstimatedSalary  Geography_France  Geography_Germany  \
0               1            50000               1.0                0.0   

   Geography_Spain  
0              0.0  


In [33]:
# scaling the input data
input_scaled = scaler.transform(input_df)

In [35]:
print(input_scaled)
print(type(input_scaled))

[[-0.53598516  0.91324755  0.10479359 -0.69539349 -0.25781119  0.80843615
   0.64920267  0.97481699 -0.87683221  1.00150113 -0.57946723 -0.57638802]]
<class 'numpy.ndarray'>


In [36]:
# predict the output using the trained model
prediction = model.predict(input_scaled)
print(prediction)

1/1 [==============================] - 0s 116ms/step
[[0.02866513]]


In [ ]:
print(prediction[0][0])
# this is prediction probability of the customer leaving the bank. We can set a threshold of 0.5 to classify whether the customer will leave or not. If the predicted probability is greater than 0.5, we can classify it as 1 (customer will leave), otherwise 0 (customer will stay).

0.028665127


In [38]:
if(prediction[0][0] > 0.5):
    print("Customer will leave the bank.")
else:
    print("Customer will stay with the bank.")

Customer will stay with the bank.
